In [1]:
import os
import time
import requests
import pandas as pd

# ---------- 0. Region boxes (10 macro-regions) ----------

# lat_min, lat_max, lon_min, lon_max
region_boxes = {
    "North America": [
        (15, 75, -170, -50),
    ],
    "Latin America & Caribbean": [
        (-60, 32, -120, -30),
    ],
    "Europe & Russia": [
        (35, 72, -10, 40),
        (55, 75, 40, 180),
    ],
    "MENA": [
        (10, 40, -20, 60),
    ],
    "Sub-Saharan Africa": [
        (-35, 10, -20, 55),
    ],
    "South Asia": [
        (5, 35, 60, 95),
    ],
    "Southeast Asia": [
        (-10, 25, 95, 135),
    ],
    "East Asia & China": [
        (20, 50, 100, 150),
    ],
    "Central Asia & Mongolia": [
        (35, 55, 50, 110),
    ],
    "Oceania": [
        (-50, 10, 110, 180),
        (-30, 10, -180, -140),
    ],
}


def region_for_latlon(lat, lon):
    """Return macro-region name for a given (lat, lon), or None."""
    for region, boxes in region_boxes.items():
        for (lat_min, lat_max, lon_min, lon_max) in boxes:
            if lat_min <= lat <= lat_max and lon_min <= lon <= lon_max:
                return region
    return None


# ---------- 1. List GSOD files for a given year ----------

def list_year_files(year, limit=None):
    """
    Return URLs for station files for the given year.
    If limit is None -> all files; otherwise cap at 'limit'.
    """
    base = f"https://www.ncei.noaa.gov/data/global-summary-of-the-day/access/{year}/"
    r = requests.get(base, timeout=60)
    r.raise_for_status()
    html = r.text

    files = []
    for part in html.split('href="')[1:]:
        href = part.split('"')[0]
        if href.endswith(".csv") or href.endswith(".csv.gz"):
            files.append(base + href)
            if limit is not None and len(files) >= limit:
                break
    return files


# ---------- 2. Load a single GSOD file ----------

def load_gsod_file(url):
    """
    Load a single GSOD station-year file:
    - keep DATE, LATITUDE, LONGITUDE, TEMP
    - add 'year' column
    Returns a small DataFrame or None.
    """
    try:
        df = pd.read_csv(url, compression="infer")
    except Exception as e:
        print("  Skipping", url, "->", e)
        return None

    keep = ["DATE", "LATITUDE", "LONGITUDE", "TEMP"]
    for col in keep:
        if col not in df.columns:
            return None

    df = df[keep].dropna(subset=["TEMP"])

    df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")
    df = df.dropna(subset=["DATE"])
    df["year"] = df["DATE"].dt.year

    return df


# ---------- 3. Process a single year ----------

def process_year(year, limit=None, min_days=200):
    """
    Process GSOD for one year:
    - list station files
    - load each
    - assign station to macro-region
    - compute station mean TEMP (F)
    - aggregate to region x year mean, convert to C

    Returns DataFrame with ['region', 'year', 'temp_mean_f', 'temp_mean_c']
    """
    start = time.time()
    urls = list_year_files(year, limit=limit)
    print(f"Year {year}: found {len(urls)} station files (limit={limit})")
    rows = []

    for i, url in enumerate(urls, start=1):
        file_name = os.path.basename(url)
        print(f"  ... file {i}/{len(urls)}: {file_name}")

        df = load_gsod_file(url)
        if df is None or df.empty:
            continue

        # skip stations with too few days of data
        if len(df) < min_days:
            continue

        lat = df["LATITUDE"].iloc[0]
        lon = df["LONGITUDE"].iloc[0]
        region = region_for_latlon(lat, lon)
        if region is None:
            continue

        mean_temp_f = df["TEMP"].mean()
        rows.append({"region": region, "year": year, "temp_mean_f": mean_temp_f})

    elapsed = time.time() - start
    print(f"Year {year}: processed loop in {elapsed:.1f} s, stations kept={len(rows)}")

    if not rows:
        return None

    df_year = pd.DataFrame(rows).groupby(["region", "year"], as_index=False)["temp_mean_f"].mean()
    df_year["temp_mean_c"] = (df_year["temp_mean_f"] - 32.0) * 5.0 / 9.0
    return df_year


# ---------- 4. Run for year and save CSV ----------

year = 1945

# Set limit=200 or 500 if Datalab times out; None = all stations
df_year = process_year(year, limit=None)

if df_year is not None:
    out_path = f"temp_region_year_{year}.csv"
    df_year.to_csv(out_path, index=False)
    print(f"Saved {out_path}")
    display(df_year)  # for Datalab to show the table
else:
    print("No data produced for year", year)


Year 1945: found 1021 station files (limit=None)
  ... file 1/1021: 03135099999.csv
  ... file 2/1021: 03301099999.csv
  ... file 3/1021: 03391099999.csv
  ... file 4/1021: 03542099999.csv
  ... file 5/1021: 03543099999.csv
  ... file 6/1021: 03546099999.csv
  ... file 7/1021: 03549099999.csv
  ... file 8/1021: 03550099999.csv
  ... file 9/1021: 03553099999.csv
  ... file 10/1021: 03554099999.csv
  ... file 11/1021: 03596399999.csv
  ... file 12/1021: 03658499999.csv
  ... file 13/1021: 03669099999.csv
  ... file 14/1021: 04018016201.csv
  ... file 15/1021: 04220099999.csv
  ... file 16/1021: 04231099999.csv
  ... file 17/1021: 04261099999.csv
  ... file 18/1021: 04270099999.csv
  ... file 19/1021: 04280199999.csv
  ... file 20/1021: 04330099999.csv
  ... file 21/1021: 04339099999.csv
  ... file 22/1021: 04361099999.csv
  ... file 23/1021: 04382099999.csv
  ... file 24/1021: 04390099999.csv
  ... file 25/1021: 07015099999.csv
  ... file 26/1021: 07147099999.csv
  ... file 27/1021: 0714

,region,year,temp_mean_f,temp_mean_c
0,East Asia & China,1945,66.622829,19.234905
1,Europe & Russia,1945,58.601769,14.778760
2,Latin America & Caribbean,1945,78.855862,26.031035
3,MENA,1945,69.454443,20.808024
4,North America,1945,55.130689,12.850383
5,Oceania,1945,73.022781,22.790434
6,South Asia,1945,77.983091,25.546162
7,Southeast Asia,1945,77.438469,25.243594
8,Sub-Saharan Africa,1945,75.712297,24.284609
